In [ ]:
import requests
import json

In [ ]:
!git checkout dataphos-workshops
!git pull

In [ ]:
participant_identification = ""

In [ ]:
if not participant_identification.islower() or not participant_identification.isalpha():
    raise ValueError("The participant identification prefix must be lowercase, without any whitespaces or special characters.")

with open("Pulumi.workshop-participant-config.yaml", 'r') as file:
    content = file.read()

content = content.replace("<participant_identification>", participant_identification)

with open("Pulumi.workshop-participant-config.yaml", 'w') as file:
    file.write(content)

In [ ]:
!pulumi stack init workshop-participant-config

In [ ]:
schema_registry_service_ip = ""

In [ ]:
url = "http://" + schema_registry_service_ip + ":8080/schemas"
headers = {"content-type": "application/json", "Accept-Charset": "UTF-8"}

r = requests.get(url, headers=headers)
print("Satus code of the response: ", r.status_code)
print(json.dumps(r.json(), indent=4))

In [ ]:
schema = {
    "description": "Schema registered manually for testing Dataphos.",
    "schema_type": "json",
    "specification": '{\r\n   "$schema":"https:\/\/json-schema.org\/draft-07\/schema",\r\n   "additionalProperties":false,\r\n   "type":"object",\r\n   "properties":{\r\n      "creation_timestamp":{\r\n         "type":"string"\r\n      },\r\n      "customer_info":{\r\n         "type":"object",\r\n         "properties":{\r\n            "age":{\r\n               "type":"integer"\r\n            },\r\n            "customer_id":{\r\n               "type":"integer"\r\n            },\r\n            "name":{\r\n               "type":"string"\r\n            },\r\n            "surname":{\r\n               "type":"string"\r\n            }\r\n         },\r\n         "required":[\r\n            "age",\r\n            "customer_id",\r\n            "name",\r\n            "surname"\r\n         ]\r\n      },\r\n      "invoice_id":{\r\n         "type":"integer"\r\n      },\r\n      "items":{\r\n         "type":"array",\r\n         "items":{\r\n            "type":"object",\r\n            "properties":{\r\n               "item_name":{\r\n                  "type":"string"\r\n               },\r\n               "price":{\r\n                  "type":"string"\r\n               }\r\n            },\r\n            "required":[\r\n               "item_name",\r\n               "price"\r\n            ]\r\n         }\r\n      }\r\n   },\r\n   "required":[\r\n      "creation_timestamp",\r\n      "customer_info",\r\n      "invoice_id",\r\n      "items"\r\n   ]\r\n}',
    "name": "Manually registered schema",
    "publisher_id": "",
    "compatibility_mode": "backward",
    "validity_mode": "full",
}

r = requests.post(url, headers=headers, json=schema)
print("Satus code of the response: ", r.status_code)
print(json.dumps(r.json(), indent=4))

## Sending messages to a ServiceBus topic

To send message to a ServiceBus topic, click the topics tab in the sidebar:

![](./Picture1.png)

Then, select input topic with __your__ name:

![](./Picture2.png)

On the left-hand size, select Service Bus Explorer:

![](./Picture3.png)

Finally, select the Send messages button:

![](./Picture4.png)

This will open up a menu on the right side. To send a message, select application/json as content type. Set the following as message body: 

```json
{
   "creation_timestamp":"2020-05-19 21:29:30 +0000 UTC",
   "customer_info":{
      "age":54,
      "customer_id":41,
      "name":"Samuel",
      "surname":"Williams"
   },
   "invoice_id":33,
   "items":[
      {
         "item_name":"flour",
         "price":"14.00"
      },
      {
         "item_name":"chips",
         "price":"28.00"
      },
      {
         "item_name":"ketchup",
         "price":"5.00"
      }
   ]
}
```

⚠️ Don't hit the Send button just yet. Add the headers so Validator will know which schema to fetch. The headers are the following (key-value pairs):

| key       | value |
|-----------|-------|
| schemaId  | 1     |
| versionId | 1     |
| format    | json  |

Now send the message. If the message validates against the schema with id and version 1-1, then the message will be visible in your storage account. On the other hand, if the message doesn't validate, it will be visible in the dead letter topic's subscription.  


In [ ]:
url = "http://" + schema_registry_service_ip + ":8080/schemas/1"
schema = {
    "description": "Schema registered manually for testing Dataphos v2.",
    "specification": '{\r\n   \"$schema\":\"https:\/\/json-schema.org\/draft-07\/schema\",\r\n   \"additionalProperties\":false,\r\n   \"type\":\"object\",\r\n   \"properties\":{\r\n      \"creation_timestamp\":{\r\n         \"type\":\"string\"\r\n      },\r\n      \"location_code\":{\r\n         \"type\":\"string\"\r\n      },\r\n      \"customer_info\":{\r\n         \"type\":\"object\",\r\n         \"properties\":{\r\n            \"age\":{\r\n               \"type\":\"integer\"\r\n            },\r\n            \"customer_id\":{\r\n               \"type\":\"integer\"\r\n            },\r\n            \"name\":{\r\n               \"type\":\"string\"\r\n            },\r\n            \"surname\":{\r\n               \"type\":\"string\"\r\n            }\r\n         },\r\n         \"required\":[\r\n            \"age\",\r\n            \"customer_id\",\r\n            \"name\",\r\n            \"surname\"\r\n         ]\r\n      },\r\n      \"invoice_id\":{\r\n         \"type\":\"integer\"\r\n      },\r\n      \"items\":{\r\n         \"type\":\"array\",\r\n         \"items\":{\r\n            \"type\":\"object\",\r\n            \"properties\":{\r\n               \"item_name\":{\r\n                  \"type\":\"string\"\r\n               },\r\n               \"price\":{\r\n                  \"type\":\"string\"\r\n               }\r\n            },\r\n            \"required\":[\r\n               \"item_name\",\r\n               \"price\"\r\n            ]\r\n         }\r\n      }\r\n   },\r\n   \"required\":[\r\n      \"creation_timestamp\",\r\n      \"customer_info\",\r\n      \"invoice_id\",\r\n      \"items\"\r\n   ]\r\n}'
}

r = requests.put(url, headers=headers, json=schema)
print("Status code of the response: ", r.status_code)
print(json.dumps(r.json(), indent=4))

In [ ]:
resubmitter_service_ip = ""

In [ ]:
resubmitter_settings = {
    "broker_id": participant_identification + "-valid-topic",
    "lb": "2024-08-06T09:45:00Z"
}
url = "http://" + resubmitter_service_ip + ":8081/range/indexer_collection?topic=" + participant_identification + "-resubmitter-topic"

r = requests.post(url, headers=headers, json=resubmitter_settings)
print("Satus code of the response: ", r.status_code)
print(json.dumps(r.json(), indent=4))

In [ ]:
!pulumi destroy --yes